# BEM layers from CHARM parcellation

## 🧪 - Experiment setup
- `charm`: CHARM head segmentation
- `t1`: T1-weighted MRI 

In [1]:
# imports
import mindmap as mm

# readers
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("simf10")

# data
charm = sr.load("seg")
ras2tkras = sr.load("ras2tkras")    # to build meshes in FreeSurfer space (expected by MNE)
vol_info = sr.load("vol-info")      # metadata for writing surfaces
bem_labels = dm.get_bem_labels()


## 🧬 - Data Processing
Convert volumes to:
- inner skull mask
- outer skull mask
- skin mask

Heavy pre-processing

In [8]:
# preprocessing

# extract labels from charm segmentation
skin_mask = mm.anatomy.parcellation.create_mask(charm, regions=bem_labels["skin"])
oskull_mask = mm.anatomy.parcellation.create_mask(charm, regions=bem_labels["outer_skull"])
iskull_mask = mm.anatomy.parcellation.create_mask(charm, regions=bem_labels["inner_skull"])

# clean volumes
skin_mask = mm.anatomy.mri.nibabel_to_ants(skin_mask)
skin_mask = skin_mask.morphology("close", radius=10)

oskull_mask = mm.anatomy.mri.nibabel_to_ants(oskull_mask)
oskull_mask = oskull_mask.morphology("close", radius=10)


iskull_mask = mm.anatomy.mri.nibabel_to_ants(iskull_mask)
iskull_mask = iskull_mask.morphology("close", radius=20)

# layer object
layers_heavy = {}
layers_heavy["skin"], interp = mm.anatomy.forward.fit_surface_to_mri_mask(skin_mask, n_smooth=20)
layers_heavy["outer skull"], interp = mm.anatomy.forward.fit_surface_to_mri_mask(oskull_mask, outer_interpolator=interp, n_smooth=20)
layers_heavy["inner skull"], _ = mm.anatomy.forward.fit_surface_to_mri_mask(iskull_mask, outer_interpolator=interp, n_smooth=20)

# convert to tkras space
layers_heavy = mm.anatomy.forward.bem_transform(layers_heavy, ras2tkras)


In [9]:
import pyvista as pv
p = pv.Plotter()
p.add_mesh(layers_heavy["skin"], style="wireframe", color="green", line_width=1)
p.add_mesh(layers_heavy["outer skull"], style="wireframe", color="black", line_width=1)
p.add_mesh(layers_heavy["inner skull"], style="wireframe", color="red", line_width=1)
p.show()

Widget(value='<iframe src="http://localhost:37499/index.html?ui=P_0x7e15ba156b90_3&reconnect=auto" class="pyvi…

In [10]:
layers = mm.anatomy.forward.build_bem_layers(charm, bem_labels, n_smooth=20)
layers = mm.anatomy.forward.bem_transform(layers, ras2tkras)

In [11]:
import pyvista as pv
p = pv.Plotter()
p.add_mesh(layers["skin"], style="wireframe", color="green", line_width=1)
p.add_mesh(layers["outer skull"], style="wireframe", color="black", line_width=1)
p.add_mesh(layers["inner skull"], style="wireframe", color="red", line_width=1)
p.show()

Widget(value='<iframe src="http://localhost:37499/index.html?ui=P_0x7e15ba02b150_4&reconnect=auto" class="pyvi…

In [12]:
from pathlib import Path

folder = Path("./bem")
folder.mkdir(parents=True, exist_ok=True)
mm.anatomy.forward.save_bem_as_freesurfer(folder, layers, vol_info, overwrite=True)

folder = Path("./bem_heavy")
folder.mkdir(parents=True, exist_ok=True)
mm.anatomy.forward.save_bem_as_freesurfer(folder, layers_heavy, vol_info, overwrite=True)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
